# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rabby0501/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Lane:** Refresh / Content Opportunity Scoring.

**Task type: scoring / ranking, built on top of supervised binary classification.** The model learns from pages whose outcome is already known ("declining" or "not declining") and then gives every page a score. The score is not the product. The product is the **ranked list**: pages sorted from most to least likely to be declining, so a reviewer can start at the top.

Why not the other types:
- **Clustering** would group pages by similarity, but it has no outcome to learn from, so it cannot tell a reviewer which pages are losing visibility.
- **Pure classification** (a yes/no per page) would throw away the ordering, and a reviewer with limited time needs the order, not 16,000 "yes" answers.
- **Ranking** is what the action needs: "check the top 50 this week."

In [1]:
import os
import numpy as np
import pandas as pd

# find the starter CSV by walking up from wherever the notebook runs (Colab, local, or work/notebooks)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# the label for my lane: a page is "declining" when its recent impressions trend is down
df["is_declining"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("Task: supervised binary classification, used as a scoring / ranking tool")
print(f"Rows: {len(df):,} | positives (declining): {df['is_declining'].sum():,} ({df['is_declining'].mean():.1%})")
print("Output per page: a score between 0 and 1 (higher = look at this page sooner)")
print("Final product: pages sorted by that score = a ranked review queue")

Task: supervised binary classification, used as a scoring / ranking tool
Rows: 30,000 | positives (declining): 16,262 (54.2%)
Output per page: a score between 0 and 1 (higher = look at this page sooner)
Final product: pages sorted by that score = a ranked review queue


## 2. Target or proxy

**Target (what I predict):** whether a page is **declining**, meaning `trend_direction == "down"`.

**Where the label comes from: a defined rule applied to an observed outcome.** `trend_pct` is the observed change in a page's search impressions over the last 30 days compared with the 30 days before. A page is labelled "down" when that change is about -20% or worse, "stable" between -20% and +20%, and "up" above +20%. The code below checks this against the data.

**This is a proxy, not the thing I really care about.** What a content team actually wants to know is "does this page need a refresh?". A page can fall in impressions for reasons a refresh will not fix (seasonality, a competitor, a Google change), and a page can need a refresh without having dropped yet. So "declining" is a **measurable stand-in** for "worth reviewing", and I will describe results that way.

**Two limits I can already see:** pages labelled `flat` or `new` have no `trend_pct`, so they all land in "not declining", which is a simplification. And `impressions_90d` covers the same period that defines the label, so any feature built from the last 30 or 60 days must be checked for leakage before I use it (that is the Week 3 leakage check).

In [2]:
# Prove the label is a rule on an observed change, not something I made up.
check = df.assign(trend_direction=df["trend_direction"].str.lower())
print("trend_pct range by trend_direction (observed change in impressions, %):")
print(check.groupby("trend_direction")["trend_pct"].agg(["count", "min", "max"]).round(1).to_string())

# Recompute the change myself from the two 30-day impression columns.
recomputed = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"].replace(0, np.nan) * 100
mask = recomputed.notna() & df["trend_pct"].notna()
print(f"\nMy recomputed % change matches trend_pct for {mask.sum():,} pages "
      f"(max absolute difference {np.abs(recomputed[mask] - df.loc[mask, 'trend_pct']).max():.2f} points).")
print(f"Pages with no trend_pct (labelled 'not declining' by default): {df['trend_pct'].isna().sum():,} ({df['trend_pct'].isna().mean():.1%})")

trend_pct range by trend_direction (observed change in impressions, %):
                 count    min      max
trend_direction                       
down             16262 -100.0    -20.0
flat                 0    NaN      NaN
new                  0    NaN      NaN
stable            5962  -20.0     20.0
up                4388   20.0  44900.0

My recomputed % change matches trend_pct for 26,612 pages (max absolute difference 0.05 points).
Pages with no trend_pct (labelled 'not declining' by default): 3,388 (11.3%)


## 3. Success metric

**Metric: Precision@50, measured on clients the model has never seen.** Of the 50 pages a ranking puts first, what share are really declining? I picked 50 because it is a realistic amount of review for one person in one week. I will also glance at Precision@20 to see how the very top behaves, but Precision@50 is the one number I will defend.

**What "good" means:** it is not enough to be above zero. In this sample about 53% of pages in a typical held-out set are declining, so a random pick of 50 pages already gets roughly 26 or 27 right. A ranking is only useful if it clearly beats that random level **and** beats a simple hand-written rule, across several different client splits and not just one lucky one.

**Why held-out clients:** pages from the same client look alike, so if a client's pages appear in both train and test the score looks better than it really is. I split by `client_id`, so a client is either all in training or all in test.

In [3]:
from sklearn.model_selection import GroupShuffleSplit

def precision_at_k(scores, labels, k):
    """Of the k highest-scored pages, what fraction are truly declining?"""
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

y = df["is_declining"].values

# The simple baseline from notebook 02: stale (>=180 days since update) x visible (>=500 impressions), ranked by impressions.
hand_rule = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int) * df["impressions_90d"]

rng = np.random.default_rng(0)
rows = []
for seed in range(10):
    train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(df, y, groups=df["client_id"]))
    y_te = y[test_idx]
    rows.append({
        "split": seed,
        "declining share in test": y_te.mean(),
        "random P@50": np.mean([precision_at_k(rng.random(len(test_idx)), y_te, 50) for _ in range(200)]),
        "hand rule P@50": precision_at_k(hand_rule.values[test_idx], y_te, 50),
    })
baseline = pd.DataFrame(rows).set_index("split")
print("Baselines on 10 client-holdout splits (30% of clients held out each time):")
print(baseline.round(3).to_string())
print("\nMean:")
print(baseline.mean().round(3).to_string())

Baselines on 10 client-holdout splits (30% of clients held out each time):
       declining share in test  random P@50  hand rule P@50
split                                                      
0                        0.486        0.481            0.56
1                        0.599        0.596            0.76
2                        0.611        0.608            0.74
3                        0.458        0.467            0.54
4                        0.455        0.451            0.42
5                        0.458        0.457            0.46
6                        0.592        0.597            0.68
7                        0.453        0.450            0.38
8                        0.608        0.610            0.68
9                        0.565        0.559            0.70

Mean:
declining share in test    0.529
random P@50                0.528
hand rule P@50             0.592


## 4. The unit of analysis, as a real dataframe

**One row = one page** (one content item). The starter file has 30,000 rows and each `content_id` appears exactly once, so there is no hidden repeat of the same page. Each row carries what was **measured about that page**, and the label I defined in section 2.

- **Features (inputs):** things known about the page itself and its search performance, such as `content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`, `search_volume` and `engagement_rate`.
- **Target (what I predict):** `is_declining`, shown as the last column below.
- **Left out on purpose:** `trend_direction`, `trend_pct` and the 30-day impression, click and session columns, because the label is computed from them.
- **Grouping column:** `client_id` is not a feature. I use it only to keep a client's pages together when splitting train and test.

The table below hides the identifier columns so no client or page codes are shown.

In [4]:
FEATURES = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "search_volume", "engagement_rate"]
TARGET = "is_declining"

print("One row = one page. content_id unique:", df["content_id"].is_unique, "| rows:", len(df), "| clients:", df["client_id"].nunique())
print(f"Model table shape: {df[FEATURES + [TARGET]].shape[0]:,} rows x {len(FEATURES)} features + 1 target\n")
df[FEATURES + [TARGET]].head(8)

One row = one page. content_id unique: True | rows: 30000 | clients: 32
Model table shape: 30,000 rows x 8 features + 1 target



,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,search_volume,engagement_rate,is_declining
0,187,20,3803,10.6,0.76,3221.0,10.0,5.88,1
1,445,25,15320,20.3,0.05,2481.0,90.0,0.00,1
2,141,20,12581,36.5,0.09,3515.0,0.0,0.00,1
3,463,22,11751,6.2,0.49,NaN,10.0,1.28,0
4,263,14,19140,44.0,0.13,2803.0,0.0,0.00,1
5,147,20,3970,8.5,0.03,3080.0,720.0,0.00,1
6,90,20,20,7.0,0.00,3059.0,0.0,0.00,1
7,445,22,1724,21.2,0.06,NaN,590.0,3.57,0


## 5. Why ML beats a fixed rule here

My hand rule asks two yes/no questions (stale? visible?) and ranks by impressions. That ignores most of what the data holds, such as a page's position, its click-through rate, its length and how its age interacts with those. Several signals act together, and the right thresholds differ from client to client, so a single if-statement is a blunt tool. A model can combine all the signals and learn how they interact.

I tested this instead of assuming it. In the cell below I trained a random forest (200 trees) on the training clients and compared its Precision@50 on the held-out clients with the hand rule and with random picks, over the same 10 client splits as section 3. In my run the model scored about 0.72 versus about 0.59 for the hand rule and about 0.53 for random, and it beat the hand rule in 9 of 10 splits.

**Careful words:** this is an observed, directional result on one anonymized sample, with a spread of roughly 0.10 across splits. It shows the model ranks better than the simple rule here. It does not show that any signal causes a decline, and since `impressions_90d` overlaps the label window, part of the gain may come from information that leaks the label. The Week 3 leakage check will test that. In notebook 02 a depth-2 tree did not clearly beat the hand rule, so "ML wins" depends on the model being rich enough, and the case has to be re-earned with a clean validation setup.

In [5]:
from sklearn.ensemble import RandomForestClassifier

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
rng = np.random.default_rng(0)
rows = []
for seed in range(10):
    train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed).split(X, y, groups=df["client_id"]))
    model = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, class_weight="balanced", random_state=0, n_jobs=-1)
    model.fit(X.iloc[train_idx], y[train_idx])
    y_te = y[test_idx]
    rows.append({
        "split": seed,
        "random P@50": np.mean([precision_at_k(rng.random(len(test_idx)), y_te, 50) for _ in range(200)]),
        "hand rule P@50": precision_at_k(hand_rule.values[test_idx], y_te, 50),
        "random forest P@50": precision_at_k(model.predict_proba(X.iloc[test_idx])[:, 1], y_te, 50),
    })
compare = pd.DataFrame(rows).set_index("split")
print(compare.round(3).to_string())
print("\nMean +/- std over 10 client splits:")
for col in compare.columns:
    print(f"  {col:<20} {compare[col].mean():.3f} +/- {compare[col].std(ddof=0):.3f}")
wins = int((compare["random forest P@50"] > compare["hand rule P@50"]).sum())
print(f"\nRandom forest beat the hand rule in {wins} of {len(compare)} client splits.")

       random P@50  hand rule P@50  random forest P@50
split                                                 
0            0.481            0.56                0.62
1            0.596            0.76                0.90
2            0.608            0.74                0.70
3            0.467            0.54                0.66
4            0.451            0.42                0.68
5            0.457            0.46                0.60
6            0.597            0.68                0.86
7            0.450            0.38                0.60
8            0.610            0.68                0.80
9            0.559            0.70                0.74

Mean +/- std over 10 client splits:
  random P@50          0.528 +/- 0.068
  hand rule P@50       0.592 +/- 0.132
  random forest P@50   0.716 +/- 0.102

Random forest beat the hand rule in 9 of 10 client splits.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.